# Assignment 7: Data Clustering using Particle Swarm Optimization
Each particle = K cluster centroids. Fitness = SSE (sum of squared distances to nearest centroid).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

In [ ]:
def sse(X, centroids):
    d = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
    return np.sum(np.min(d, axis=1) ** 2)

def assign(X, centroids):
    d = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
    return np.argmin(d, axis=1)

def run_pso(X, K, n_particles, n_iter, w, c1, c2):
    np.random.seed(42)
    dim = X.shape[1]
    lo, hi = X.min(axis=0), X.max(axis=0)
    pos = np.random.uniform(lo, hi, size=(n_particles, K, dim))
    vel = np.zeros_like(pos)
    pbest = pos.copy()
    pbest_score = np.array([sse(X, p) for p in pos])
    gbest = pbest[np.argmin(pbest_score)].copy()
    gbest_score = pbest_score.min()
    history = []

    for it in range(n_iter):
        for i in range(n_particles):
            r1, r2 = np.random.rand(K, dim), np.random.rand(K, dim)
            # Velocity and position update
            vel[i] = w * vel[i] + c1 * r1 * (pbest[i] - pos[i]) + c2 * r2 * (gbest - pos[i])
            pos[i] = pos[i] + vel[i]

            score = sse(X, pos[i])
            if score < pbest_score[i]:              # update personal best
                pbest[i], pbest_score[i] = pos[i].copy(), score
                if score < gbest_score:             # update global best
                    gbest, gbest_score = pos[i].copy(), score
        history.append(gbest_score)
        if (it + 1) % 10 == 0:
            print(f"Iteration {it + 1:3d}: best SSE = {gbest_score:.2f}")
    return gbest, gbest_score, history

In [ ]:
K, n_points = 3, 300
n_particles, n_iter, w, c1, c2 = 20, 100, 0.7, 1.5, 1.5
X, _ = make_blobs(n_samples=n_points, centers=K, cluster_std=0.8, random_state=42)
gbest, gbest_score, history = None, None, None

while True:
    print("\n========= PSO CLUSTERING MENU =========")
    print(f"Points = {n_points}, K = {K}, Particles = {n_particles}, Iterations = {n_iter}, "
          f"w = {w}, c1 = {c1}, c2 = {c2}")
    print("1. Generate new dataset")
    print("2. Set PSO parameters")
    print("3. Run PSO clustering")
    print("4. Show final centroids and SSE")
    print("5. Compare with K-Means")
    print("6. Plot clusters and convergence")
    print("0. Exit")
    ch = input("Enter your choice: ").strip()
    try:
        if ch == '1':
            n_points = int(input("Number of points: "))
            K = int(input("Number of clusters K: "))
            X, _ = make_blobs(n_samples=n_points, centers=K, cluster_std=0.8, random_state=42)
            gbest = None
        elif ch == '2':
            n_particles = int(input("Number of particles: "))
            n_iter = int(input("Number of iterations: "))
            w  = float(input("Inertia weight w: "))
            c1 = float(input("Cognitive coefficient c1: "))
            c2 = float(input("Social coefficient c2: "))
        elif ch == '3':
            gbest, gbest_score, history = run_pso(X, K, n_particles, n_iter, w, c1, c2)
        elif ch == '4':
            if gbest is None: print("Run PSO first (option 3)."); continue
            print("Final centroids:\n", np.round(gbest, 2))
            print(f"Final SSE: {gbest_score:.2f}")
        elif ch == '5':
            if gbest is None: print("Run PSO first (option 3)."); continue
            km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
            print(f"PSO SSE    : {gbest_score:.2f}")
            print(f"K-Means SSE: {km.inertia_:.2f}")
        elif ch == '6':
            if gbest is None: print("Run PSO first (option 3)."); continue
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.scatter(X[:, 0], X[:, 1], c=assign(X, gbest), s=15)
            plt.scatter(gbest[:, 0], gbest[:, 1], c='red', marker='X', s=200)
            plt.title('PSO Clustering')
            plt.subplot(1, 2, 2)
            plt.plot(history); plt.title('Convergence (SSE)'); plt.xlabel('Iteration')
            plt.tight_layout(); plt.show()
        elif ch == '0':
            print("Exiting...")
            break
        else:
            print("Invalid choice")
    except ValueError:
        print("Please enter valid numbers")